# 10 自定义 attention：监控、只读 top-K、改 KV 布局

在阅读过 00、06 和 07 之后，如果你关心 attention 层面的推理算法——比如统计注意力落在哪些位置上的注意力监控、训练无关的稀疏读取（如 Quest 按页挑选、只看部分位置），或者改变 KV 的存储布局（滑动窗口、保留 prompt 加最近窗口、跨层共享 KV）——你会发现这些算法有一个共同点：它们都不改 q/k/v 的投影，只改“这一层读 KV 的方式”或“KV 怎么存”。但在 vLLM 里，attention 是一个融合好的 kernel，KV 分页存放、由调度器分配和回收；想要改动它，通常需要修改 attention 后端或模型代码。

`@pllm.custom_attention` 只替换指定层的 attention 读法：原生的投影和 KV 写入照常进行，方法拿到当前层（已加 RoPE 的）`q` 和一个只读的分页 KV 视图 `kv`，用 `kv.attend(...)` 调用原生 kernel（可加窗口或每个头各自的位置选择），返回 attention 输出，还可以多返回一个 `aux` 张量给后面的方法读。同时，同一个装饰器上的 `keep=`、`window=`、`sink=`、`source=` 参数用来声明 KV 怎么存，具体的存储由 vLLM 自己的滑动窗口、R-SWA 或 KV 共享机制负责分配和回收。

在开始之前，有几项事实、限制和代价需要先说明清楚：
- Qwen2.5-0.5B 没有按这些布局训练过。跨层共享和偶数层滑动窗口在这里会把答案弄坏（实测见第三步）；这个示例演示的是接法、检查和代价，不演示生成质量。
- 只读 top-K（选择读取）省下的是读取的数据量，不省 KV 显存。这里的页打分直接用 PyTorch 编写，每层每步都要把所有页的 key 读一遍，实测吞吐只有原生的 7%：它用来说明语义，不是提速的写法。真正提速需要把页的 min/max 增量维护，并写成自定义 kernel（见示例 09 的做法）。
- 按分数真正释放 KV 显存（例如 H2O、SnapKV 这类驱逐算法）在 pLLM 中不支持（`Score-based memory eviction is unsupported`）。原因在于：vLLM 假设每个请求的 KV 是从头开始连续的一段，位置和块表都由“已经算了多少 token”推出，块表只增不减；要在中间挖洞，需要修改 vLLM 的这些底层约定。
- 运行限制：要求单卡（`tensor_parallel_size=1`）、未量化 KV 的 FlashAttention 2 后端、关闭前缀缓存；静态布局需要 MRv2 runner，且不能和 `PositionState` 或 `@pllm.custom_forward` 一起使用。

本示例分三步带你做实验（实测环境：RTX 4090、vLLM 0.27.1、Qwen2.5-0.5B-Instruct、bf16）：
1. **监控**：读出注意力落在最近位置上的比例，不改变模型输出。
2. **只读 top-K**：实现 Quest 式的按页挑选读取，并对照“同样预算下只读最近的位置”。
3. **KV 布局**：对比滑动窗口、保留 prompt 加窗口、跨层共享这几种布局各自省下多少 KV 显存，以及吞吐是多少。

为了方便观察不同读法和布局对长上下文的影响，我们先准备一段约 1500 token 的测试长 prompt：在一篇博物馆介绍中间埋入一句 `"The secret code for the blue door is 7361."`，然后向模型提问密码是多少。下面先用原生 vLLM 回答一次作为基准。

In [1]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # 只显示 vLLM 的报错，省掉启动日志

from collections import defaultdict
from dataclasses import dataclass, replace

import torch
from transformers import AutoTokenizer
from vllm import SamplingParams

import pllm
from pllm import testing
from pllm.benchmark import throughput

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# 2048 token 的上下文；分块 prefill，每步最多 256 个 token；pLLM 的 CUDA graph 捕获预算 1 GiB
# （执行选项只作用于带负载的引擎，同一个配置也可以起原生 vLLM）。
CONFIG = pllm.ModelConfig(MODEL, max_model_len=2048, max_num_seqs=8, max_num_batched_tokens=256,
                          enable_chunked_prefill=True, gpu_memory_utilization=0.4, seed=0,
                          execution=pllm.ExecutionOptions(cuda_graph_pool_budget_bytes=1 << 30))

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
filler = ("The museum opens at nine in the morning and closes at six in the evening. Visitors can buy tickets at the "
          "front desk or online. The cafe on the second floor serves coffee, tea and sandwiches. ")
passage = filler * 6 + "The secret code for the blue door is 7361. " + filler * 30
question = "What is the secret code for the blue door? Answer with the number only."
prompt = tokenizer.apply_chat_template([{"role": "user", "content": passage + "\n\n" + question}], tokenize=False,
                                       add_generation_prompt=True)
greedy = SamplingParams(max_tokens=12, temperature=0.0)
print("prompt 长度:", len(tokenizer(prompt).input_ids), "token")


def answer(llm) -> str:
    return llm.generate([prompt], greedy, use_tqdm=False)[0].outputs[0].text


with pllm.make_llm(CONFIG) as plain:
    print("原生 vLLM 的回答:", repr(answer(plain)))

prompt 长度: 1500 token
原生 vLLM 的回答: '7361'


原生 vLLM 在这段 1500 token 的 prompt 上回答出了 `'7361'`。接下来进入第一步：**我们要如何在不改变模型输出的前提下，监控某一层的注意力有多大比例落在了最近的位置上？**

这里我们关心的是如何在 attention 计算过程中提取中间统计量并导出到主机。`kv.attend(q, *, window=0, topk=None)` 会返回 `(输出 [行, 头, 头维], float32 LSE [行, 头])`，不带选择参数时调用的就是原生的因果 kernel；其中的 LSE 是 log-sum-exp，即 softmax 归一化常数取对数。

下面的 `RecentMass` 负载把自定义 attention 方法 `split` 只装在第 12 层：
- 先调用一次 `kv.attend(q)` 完成原生的读取，拿到正常输出 `out` 和全序列的 `lse`；
- 再调用一次 `kv.attend(q, window=64)`，只读最近 64 个位置（`window` 从当前 token 往回数，含当前 token），拿到局部窗口的 `near`（即局部 LSE）；
- 将两个 LSE 相减再取指数 `(near - lse).exp()`，算出的就是“最近 64 个位置分到的注意力占比”，对注意力头取平均后作为第二个输出（`aux`）返回；
- 第一个返回值依然用第一次算出的原生结果 `out`，因此模型行为保持不变。

接着，`@pllm.export` 装饰的 `report` 方法通过 `pllm.attention(12, "aux")` 读取第 12 层自定义 attention 返回的这个 `aux` 张量（在导出方法里读到的是被采样的行；如果写 `"lse"` 则会读取该层第一次 `kv.attend` 的 LSE），每步将其导出到主机，最后由 `on_export` 回调按请求（`prompt_index`）收集起来。

我们同时跑两个请求来做对照：一个是刚才 1500 token 的博物馆长 prompt，另一个是很短的 `"What is 2 + 3?"`。

In [3]:
@dataclass(frozen=True)
class RecentMass(pllm.Workload):
    recent: int = 64

    @pllm.custom_attention(layers=(12,))
    def split(self, q: torch.Tensor, kv: pllm.model.PagedKV) -> tuple[torch.Tensor, torch.Tensor]:
        out, lse = kv.attend(q)                     # 原生的读法，同时拿到 log-sum-exp
        _, near = kv.attend(q, window=self.recent)  # 只读最近 recent 个位置
        return out, (near - lse).exp().mean(-1, keepdim=True)  # aux：最近位置分到的注意力占比，对头取平均

    @pllm.export(name="recent_mass", max_rows=8, max_cols=1)
    def report(self, share: torch.Tensor = pllm.attention(12, "aux")) -> torch.Tensor:
        return share

In [4]:
shares: dict[int, list[float]] = defaultdict(list)


@dataclass(frozen=True)
class RecordedMass(RecentMass):
    def on_export(self, event: pllm.io.TraceEvent) -> None:
        for identity, row in zip(event.identities, event.tensor().tolist()):
            shares[identity.prompt_index].append(round(row[0], 3))


short = tokenizer.apply_chat_template([{"role": "user", "content": "What is 2 + 3?"}], tokenize=False,
                                      add_generation_prompt=True)
with pllm.make_llm(CONFIG, workload=RecordedMass()) as llm:
    outputs = llm.generate([prompt, short], greedy, use_tqdm=False)
    print("eager 原因:", repr(llm.diagnostics().eager_reason))
for index, output in enumerate(outputs):
    print(f"prompt {index}: {output.outputs[0].text!r}")
    print("    最近 64 个位置的注意力占比:", shares[index])

eager 原因: ''
prompt 0: '7361'
    最近 64 个位置的注意力占比: [0.633, 0.588, 0.666, 0.757, 0.586]
prompt 1: '2 + 3 equals 5.'
    最近 64 个位置的注意力占比: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


先看运行结果：`eager 原因` 为空字符串 `''`，说明这个监控负载在 CUDA graph 里运行。长 prompt（`prompt 0`）的回答依然是 `'7361'`；在它生成的每个 token 上，第 12 层最近 64 个位置分到了 `0.586` 到 `0.757` 的注意力，其余注意力落在更早的位置。短 prompt（`prompt 1`）回答了 `'2 + 3 equals 5.'`，每一步的占比都是 `1.0`——因为它的 prompt 只有 37 个 token，加上生成的内容总长也不到 64 个 token，全部位置都在“最近 64 个”的范围里。至于这个监控是否真的没有改变生成的 token，我们会在最后一步用 `check_on_gpu` 做逐位验证。

接下来进入第二步：**如果每层每个头只允许读固定预算的少量位置，按页挑选高分位置（Quest）和直接只读最近的位置有什么区别？**

Quest（Tang et al., 2024）的做法用白话概括就是：在 vLLM 中，KV 按页（这里每页 16 个 token）存放。对每一页，记下这一页里所有 key 在每个维度上的最小值和最大值；对于当前的查询向量 $q$，$\sum_d \max(q_d \cdot \text{min}_d, q_d \cdot \text{max}_d)$ 就是 $q \cdot k$ 在这一页里可能取到的最大值的上界。每个注意力头各自挑出上界最大的几页，只对这几页做 attention。

在下面的代码里：
- `page_scores` 用 PyTorch 直接从 `kv.key_cache` 和 `kv.block_table` 算出每一行、每个头、每一页的上界（Qwen2.5-0.5B 使用 GQA，14 个 query 头按 2 个 KV 头分组）。
- `QuestRead` 装在第 2 到 23 层上（第 0、1 层照常读全部位置）。它先把当前位置还没写到的页屏蔽成 `-inf`，然后每个头各自取上界最大的 `pages` 页，换算成 token 的逻辑位置交给 `kv.attend(q, topk=...)`（`topk` 支持形状 `[行, k]` 或 `[行, 头, k]`，不足处可用 `-1` 补齐，不同头、不同层可以有不同预算）。
- 为了对照，当 `recent_only=True` 时，我们在同样的预算（8 页 = 128 个 token）下改为调用 `kv.attend(q, window=128)`，即只读最近的 128 个位置。另外我们还定义了 `NativeRead`，每层读全部位置，既作为这里的对照，也作为下一步比较 KV 布局的基准。

下一个单元格先在 CPU 上运行 `pllm.explain` 查看读取开销估算和契约检查，随后在 GPU 上测试它们能否答出密码问题。

In [5]:
def page_scores(q: torch.Tensor, kv: pllm.model.PagedKV) -> torch.Tensor:
    """Quest：用每页 key 在各维上的最小/最大值，给 q·k 在这一页里的最大可能值一个上界。"""
    keys = kv.key_cache[kv.block_table[kv.requests].long()].float()  # [行, 页, 页内位置, kv 头, 头维]
    low, high = keys.amin(2), keys.amax(2)                           # [行, 页, kv 头, 头维]
    rows, heads, dim = q.shape
    g = q.float().reshape(rows, low.shape[2], heads // low.shape[2], dim)  # GQA：按 kv 头分组
    bound = (torch.einsum("rkgd,rpkd->rkgp", g.clamp(min=0), high)
             + torch.einsum("rkgd,rpkd->rkgp", g.clamp(max=0), low))
    return bound.reshape(rows, heads, -1)                            # [行, 头, 页]


@dataclass(frozen=True)
class NativeRead(pllm.Workload):
    """对照组：每层照常读全部位置，KV 照常全部保存。"""

    @pllm.custom_attention()
    def read(self, q: torch.Tensor, kv: pllm.model.PagedKV) -> torch.Tensor:
        return kv.attend(q)[0]


@dataclass(frozen=True)
class QuestRead(pllm.Workload):
    pages: int = 8          # 每个头读几页（每页 16 个 token）
    recent_only: bool = False  # 对照组：同样的预算，只读最近的位置

    @pllm.custom_attention(layers=tuple(range(2, 24)))
    def read(self, q: torch.Tensor, kv: pllm.model.PagedKV) -> torch.Tensor:
        block = kv.key_cache.shape[1]
        if self.recent_only:
            return kv.attend(q, window=self.pages * block)[0]
        scores = page_scores(q, kv)
        page = torch.arange(scores.shape[-1], device=q.device)
        scores = scores.masked_fill(page * block > kv.positions[:, None, None], -float("inf"))  # 还没写到的页
        chosen = scores.topk(min(self.pages, scores.shape[-1]), dim=-1).indices            # [行, 头, pages]
        tokens = (chosen[..., None] * block + torch.arange(block, device=q.device)).flatten(2)
        return kv.attend(q, topk=tokens)[0]  # 每个头只读自己选中的位置

In [6]:
for workload in (NativeRead(), QuestRead()):
    for line in str(pllm.explain(workload, CONFIG)).splitlines():
        if line.startswith("attention layer 2:"):
            print(f"{type(workload).__name__:10s}", line)
print(testing.check(QuestRead(), CONFIG))

NativeRead attention layer 2: custom `read`; all positions natively; 7340032 FLOP/token, <= 1048576 KV bytes/token [estimate at max_model_len; per-head budget upper bound]
QuestRead  attention layer 2: custom `read`; topk <= 128/head; 458752 FLOP/token, <= 458752 KV bytes/token [estimate at max_model_len; per-head budget upper bound]
status  contract / reason
PASS    tensor_parallel_size=1: admission
PASS    compile
PASS    custom-attention method math / CPU paged KV schedules
PASS    row permutation
PASS    request isolation
PASS    mixed prefill/decode
PASS    chunking / tokens=
PASS    preempted
SKIP    publish_lag: no row method reads versioned state
SKIP    neutral: pass neutral= with the author's disabled settings


In [7]:
for workload in (QuestRead(pages=8), QuestRead(pages=8, recent_only=True), QuestRead(pages=2)):
    with pllm.make_llm(CONFIG, workload=workload) as llm:
        print(f"{workload}: {answer(llm)!r}")

QuestRead(pages=8, recent_only=False): '7361'
QuestRead(pages=8, recent_only=True): ''
QuestRead(pages=2, recent_only=False): '9'


我们先看 `explain` 在 CPU 上给出的估算（上下文按 `max_model_len=2048` 计算）：
- `NativeRead` 读全部位置时，第 2 层每个 token 最多读 `1048576` 字节 KV，计算量为 `7340032` FLOP/token；
- `QuestRead` 每个头读 128 个位置（`topk <= 128/head`）时，计算量降到 `458752` FLOP/token，但每个 token 最多要读 `458752` 字节 KV。

为什么 128 个位置只是 2048 的 1/16，KV 读取量的上限却占到了约 44%（`458752 / 1048576 ≈ 0.44`）？因为这个模型的 14 个 query 头共享 2 个 KV 头，当每个 query 头各选各的位置时，同一个 KV 头的数据会被不同的 query 头各读一遍。`explain` 在写代码阶段就把这个代价直接摆了出来。

接着看 `testing.check` 的 CPU 契约检查：自定义 attention 方法的数学逻辑在各种分页 KV 调度下保持一致，行置换（`row permutation`）、请求隔离（`request isolation`）、prefill/decode 混合（`mixed prefill/decode`）、分块（`chunking / tokens=`）以及抢占重算（`preempted`）全部 `PASS`。

再看 GPU 上的回答结果：
- `QuestRead(pages=8, recent_only=False)` 每头挑 8 页（128 个 token）时，仍然答出了 `'7361'`；
- `QuestRead(pages=8, recent_only=True)` 在同样 128 个 token 的预算下只读最近位置，回答是空字符串 `''`（第一个生成的 token 就是结束符），因为包含密码的那句话在最近 128 个位置之外；
- `QuestRead(pages=2, recent_only=False)` 把预算缩减到每头只读 2 页时，回答成了 `'9'`。

请注意，这只是一个 prompt 上的功能演示，不是质量评测。此外还要再次提醒：只读 top-K 省下的是读取量，不会节省 KV 显存。

接下来进入第三步：**如果我们希望真正减少每个请求占用的 KV 显存，应该怎么声明 KV 存储布局？不同布局各自能省多少显存、对模型输出有什么影响？**

我们在下面的代码中对比四个负载，它们的方法体全都只是 `return kv.attend(q)[0]`，区别只在于 `@pllm.custom_attention` 装饰器里的存储布局声明（窗口裁剪和共享都由存储布局本身负责）：
1. `NativeRead`：默认全部保存。
2. `LocalGlobal`：偶数层声明 `keep="window", window=256`，只保留最近 256 个 token 的 KV，更早的块由 vLLM 的滑动窗口机制自动释放；奇数层照常保存全部 KV。
3. `PromptPlusWindow`：偶数层在 `keep="window", window=256` 的基础上加上 `sink=True`，保留整个 prompt 再加上最近 256 个生成的 token（对应 vLLM 的 R-SWA）；奇数层照常。
4. `CrossLayerShared`：奇数层声明 `keep="none", source=1`，这一层不分配自己的 KV，直接读取下面第 1 层（即前一层偶数层）的 KV。

不需要启动 GPU 引擎，`pllm.explain` 在 CPU 上就能算出每个请求最多需要多少 KV 显存（`explain` 算出的这个值是 vLLM 自己的分配上限，包含了正在调度中的 token）。

In [8]:
@dataclass(frozen=True)
class LocalGlobal(pllm.Workload):
    """偶数层只保留最近 256 个 token 的 KV（滑动窗口），奇数层照常。"""

    @pllm.custom_attention(layers=tuple(range(0, 24, 2)), keep="window", window=256)
    def read(self, q: torch.Tensor, kv: pllm.model.PagedKV) -> torch.Tensor:
        return kv.attend(q)[0]


@dataclass(frozen=True)
class PromptPlusWindow(pllm.Workload):
    """偶数层保留整个 prompt，加上最近 256 个生成的 token。"""

    @pllm.custom_attention(layers=tuple(range(0, 24, 2)), keep="window", window=256, sink=True)
    def read(self, q: torch.Tensor, kv: pllm.model.PagedKV) -> torch.Tensor:
        return kv.attend(q)[0]


@dataclass(frozen=True)
class CrossLayerShared(pllm.Workload):
    """奇数层不存 KV，直接读下面一层的 KV（CLA 式跨层共享）。"""

    @pllm.custom_attention(layers=tuple(range(1, 24, 2)), keep="none", source=1)
    def read(self, q: torch.Tensor, kv: pllm.model.PagedKV) -> torch.Tensor:
        return kv.attend(q)[0]


LAYOUTS = (NativeRead(), LocalGlobal(), PromptPlusWindow(), CrossLayerShared())
for workload in LAYOUTS:
    per_request = sum(capacity.bytes for capacity in pllm.explain(workload, CONFIG).kv_capacity)
    print(f"{type(workload).__name__:17s}: 每个请求最多 {per_request / 2**20:5.1f} MiB KV")

NativeRead       : 每个请求最多  24.0 MiB KV
LocalGlobal      : 每个请求最多  16.6 MiB KV
PromptPlusWindow : 每个请求最多  24.0 MiB KV
CrossLayerShared : 每个请求最多  12.0 MiB KV


In [9]:
for workload in LAYOUTS:
    with pllm.make_llm(CONFIG, workload=workload) as llm:
        concurrency = llm.diagnostics().kv_cache_max_concurrency  # vLLM 按实际分配的 KV 池算出
        print(f"{type(workload).__name__:17s}: 可同时容纳 {concurrency:6.1f} 个 2048-token 请求；"
              f"回答 {answer(llm)!r}")

NativeRead       : 可同时容纳  358.0 个 2048-token 请求；回答 '7361'
LocalGlobal      : 可同时容纳  517.8 个 2048-token 请求；回答 '3'
PromptPlusWindow : 可同时容纳  358.0 个 2048-token 请求；回答 '7361'
CrossLayerShared : 可同时容纳  716.0 个 2048-token 请求；回答 'arga textDecorationugi-ingenSalesproj:oランキング étapelessKey'


把 `explain` 在 CPU 上的预估和 vLLM 启动后的实际分配放在一起看：
- `explain` 算出四个负载每个请求最多占用的 KV 显存分别是：`NativeRead` `24.0 MiB`、`LocalGlobal` `16.6 MiB`、`PromptPlusWindow` `24.0 MiB`、`CrossLayerShared` `12.0 MiB`。
- 启动引擎后，`llm.diagnostics().kv_cache_max_concurrency` 给出 vLLM 按真实分配的 KV 池算出的并发容量，即能同时装下几个 2048-token 的请求：分别是 `358.0`、`517.8`、`358.0` 和 `716.0` 个。
- 对比比例可以看到，`LocalGlobal` 的并发容量是 `NativeRead` 的 1.45 倍（`24.0 / 16.6 ≈ 1.45`，`517.8 / 358.0 ≈ 1.45`），`CrossLayerShared` 约是 2 倍（`716.0 / 358.0 = 2.00`），`explain` 的预估与 vLLM 的实际分配一致。
- 为什么 `PromptPlusWindow` 的单请求上限还是 `24.0 MiB`、并发容量（`358.0`）也和 `NativeRead` 一样，一点都没省？因为分配上限是按最坏情况计算的：一个请求的 prompt 本身就可能占满 2048 个 token，而 `sink=True` 要求保留整个 prompt；它真正节省显存的场景是生成序列很长时 prompt 之后超出窗口的那部分。

再看这四种布局在密码问题上的回答：
- `LocalGlobal` 回答成了 `'3'`，`CrossLayerShared` 输出了乱码 `'arga textDecorationugi-ingenSalesproj:oランキング étapelessKey'`。正如开头说明的，Qwen2.5-0.5B 没有按局部滑动窗口或跨层共享 KV 的方式训练过，直接改布局会把答案弄坏。
- `PromptPlusWindow` 仍然回答 `'7361'`：因为这次生成的长度只有 12 个 token，加上保留了完整 prompt，实际上没有任何 KV 被丢掉。

看完了显存容量，下一步我们测一下**这些自定义读法和 KV 布局在批量并发生成时的吞吐表现如何**。测试方式是：同时跑 128 个并发请求（8 个 prompt × 每个采样 16 个样本 × 每个生成 256 token），用 `pllm.benchmark.throughput` 测 token/s：预热一轮，再计时三轮取中位数，每轮计整次 `generate` 的时间（含 prefill）。

In [10]:
many = [f"Explain the mathematical idea numbered {i}:" for i in range(8)]
sampling = SamplingParams(n=16, max_tokens=256, temperature=0.8, top_p=0.95, ignore_eos=True, seed=7)
FAST = replace(CONFIG, max_num_seqs=128)

rates = {}
with pllm.make_llm(FAST) as plain:
    rates["原生 vLLM"] = throughput(plain, many, sampling)
for workload in (NativeRead(), QuestRead(pages=8), LocalGlobal(), CrossLayerShared()):
    with pllm.make_llm(FAST, workload=workload) as llm:
        rates[type(workload).__name__] = throughput(llm, many, sampling)
for name, rate in rates.items():
    print(f"{name:17s}: {rate:6.0f} token/s，为原生 vLLM 的 {rate / rates['原生 vLLM']:.0%}")

原生 vLLM          :  39812 token/s，为原生 vLLM 的 100%
NativeRead       :  39182 token/s，为原生 vLLM 的 98%
QuestRead        :   2827 token/s，为原生 vLLM 的 7%
LocalGlobal      :  38736 token/s，为原生 vLLM 的 97%
CrossLayerShared :  38242 token/s，为原生 vLLM 的 96%


来看这次运行的吞吐（基准 `原生 vLLM` 为 `39812 token/s`）：
- 换成什么都不改的自定义 attention（`NativeRead`）后，吞吐为 `39182 token/s`，是原生 vLLM 的 `98%`。
- 改用存储布局 `LocalGlobal`（`38736 token/s`）和 `CrossLayerShared`（`38242 token/s`），吞吐分别是原生 vLLM 的 `97%` 和 `96%`。在这个吞吐测试里，每个序列总长只有约 270 个 token，大小为 256 的窗口几乎没有丢掉什么内容，所以改布局在这里不会带来计算提速；它们省下的是上一步看到的 KV 显存。
- `QuestRead` 的吞吐只有 `2827 token/s`，为原生 vLLM 的 `7%`。原因我们在开头已经交底过：这里的 `page_scores` 是用 PyTorch 写的，每层每步都要把所有页的 key 完整读出来算一次 min/max，开销很大；它只用来说明按页选择读取的语义，要想真正提速，必须把每页的 min/max 做增量维护并写成自定义 kernel（如示例 09 所示）。

最后一步，我们回到最初的监控负载 `RecentMass`，用 `testing.check_on_gpu` 在 GPU 上做一次完整的契约检查。这里把 `neutral` 参数传为 `RecentMass()` 自己：因为监控负载的设计目标就是完全不改变模型的生成结果，所以直接拿它自己当“关闭修改”的对照版本，与原生 vLLM 逐位比对生成的 token。

In [11]:
print(testing.check_on_gpu(RecentMass(), CONFIG, neutral=RecentMass()))

status  contract / reason
PASS    explain/probe
PASS    host syncs: eight steady decode steps
PASS    preempted: fixed per-token inputs, token IDs and terminal request state
PASS    neutral: greedy and seeded sampling bitwise


GPU 契约检查的四项全部 `PASS`：
- `explain/probe` 通过；
- `host syncs: eight steady decode steps`：连续 8 个稳定的 decode 步内没有发生 GPU 到 CPU 的同步；
- `preempted`：请求被强制驱逐后再重算，每个 token 的输入、生成的 token ID 以及请求的终止状态都能复现；
- `neutral: greedy and seeded sampling bitwise`：无论是贪心解码还是带随机种子的采样，`RecentMass` 生成的 token 都与原生 vLLM 逐位相同。

### 小结

- `@pllm.custom_attention` 只替换指定层的 attention 读法，原生的 q/k/v 投影和 KV 写入照常进行。
- `kv.attend` 返回原生 kernel 的输出和 float32 LSE，支持通过 `window=` 指定局部窗口，或通过 `topk=` 为每个头指定各自要读的逻辑位置；方法可以额外返回
一个 `aux` 张量，供后续方法通过 `pllm.attention(layer, "aux")` 读取（或用 `"lse"` 读取第一次 `kv.attend` 的 LSE）。
- KV 的存储布局（`keep="window"`、`window=`、`sink=True`、`keep="none"`、`source=`）直接写在装饰器参数里，底层存储的分配与回收交给 vLLM 自己的机制负责；`pllm.explain` 在启动引擎前就能在 CPU 上给出每个请求的 KV 显存上限。
- 区分两类做法的边界：只读 top-K（选择读取）省的是读取量，不省 KV 显存；而按分数动态驱逐并释放 KV 显存的做法（如 H2O、SnapKV）pLLM 不支持。
- 适用限制：当前要求单卡（`tensor_parallel_size=1`）、未量化 KV 的 FlashAttention 2 后端、关闭前缀缓存；静态布局需要 MRv2 runner，且不能与 `PositionState` 或 `@pllm.custom_forward` 同时使用。

### 下一步

下一篇 [11 查表记忆](11_lookup_memory.ipynb) 将介绍如何按 token id 查一张大表，并由你决定把这张表放在 GPU 显存还是主机内存里。